# Equations

Costs are price at timestamp $t$ mutiplied by sum of energy consumption/producton by each device at the given timestamp.

$$costs = \sum_{t \in T}{\left[ P_t \cdot \sum_{d \in D} E_{d,t} \right]}$$

Overall constraint:
- Max power should not exceed line capacity
$$ \left|\sum_{d \in D} E_{d,t}\right| \le E_{max}: \forall t \in T$$


# Classes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from dataclasses import dataclass, field
from typing import Optional, Protocol
from ortools.sat.python import cp_model

In [ ]:
class ModelElement(Protocol):
    v_power: list[cp_model.IntVar]

    def add_to_model(self, time_horizon: int, model: cp_model.CpModel): ...
    def evaluate(self, solver: cp_model.CpSolver): ...

In [ ]:
@dataclass
class Device(ModelElement):
    name: str
    power_profile: list[int]
    """ units: watt-hours"""
    lim_start: Optional[int] = None
    lim_end: Optional[int] = None

    @property
    def len(self):
        return len(self.power_profile)

    def add_to_model(self, time_horizon: int, model: cp_model.CpModel):
        # start-end variable
        self.v_start = model.new_int_var(0, time_horizon, f"{self.name}_ts_start")
        self.v_end = self.v_start + self.len
        model.add(self.v_end <= time_horizon)

        # boolean start variables
        active = [
            model.new_bool_var(f"{self.name}_active_{ts}")
            for ts in range(time_horizon - self.len + 1)
        ]
        model.add_exactly_one(active)

        # power profile variables
        max_power = max(self.power_profile)
        self.v_power = [
            model.new_int_var(0, max_power, f"{self.name}_power_{ts}")
            for ts in range(time_horizon)
        ]

        for i, act in enumerate(active):
            model.add(self.v_start == i).only_enforce_if(act)
            for k, v in enumerate(self.power_profile):
                model.add(self.v_power[i + k] == v).only_enforce_if(act)

        # constraints
        if self.lim_start:
            model.add(self.v_start >= self.lim_start)

        if self.lim_end:
            model.add(self.v_end < self.lim_end)

    def evaluate(self, solver: cp_model.CpSolver):
        self.start = solver.value(self.v_start)
        self.end = solver.value(self.v_end)
        self.power = np.array([solver.value(p) for p in self.v_power])


@dataclass
class Battery(ModelElement):
    name: str
    capacity: int
    max_inflow: int
    max_outflow: int = 0
    start_level: int = 0
    end_level: Optional[int] = None

    def __post_init__(self):
        assert self.max_inflow > 0
        if self.max_outflow == 0:
            self.max_outflow = -self.max_inflow

        assert self.max_outflow < 0

    def add_to_model(self, time_horizon: int, model: cp_model.CpModel):
        self.v_power = [
            model.new_int_var(
                self.max_outflow, self.max_inflow, f"{self.name}_power_{ts}"
            )
            for ts in range(time_horizon)
        ]
        self.v_level = [
            model.new_int_var(0, self.capacity, f"{self.name}_level_{ts}")
            for ts in range(time_horizon)
        ]

        for ts in range(time_horizon):
            # times 4 due to 15 min intervals
            model.add(
                4 * self.start_level + sum(self.v_power[:ts]) == 4 * self.v_level[ts]
            )

        if self.end_level:
            model.add(self.v_level[-1] >= self.end_level)

    def evaluate(self, solver: cp_model.CpSolver):
        self.power = np.array([solver.value(p) for p in self.v_power])
        self.level = np.array([solver.value(l) for l in self.v_level])


@dataclass
class FixedDevice(ModelElement):
    name: str
    power_profile: list[int]
    """ units: watt-hours"""

    def add_to_model(self, time_horizon: int, model: cp_model.CpModel):
        self.v_power = [
            model.new_int_var(
                self.power_profile[ts],
                self.power_profile[ts],
                f"{self.name}_power_{ts}",
            )
            for ts in range(time_horizon)
        ]

    def evaluate(self, solver: cp_model.CpSolver):
        self.power = np.array([solver.value(p) for p in self.v_power])


@dataclass
class HouseholdModel:
    max_power: int
    prices: list[float]

    def __post_init__(self):
        self.elements: list[ModelElement] = []
        self.time_horizon = len(self.prices)

    def add(self, element: ModelElement):
        self.elements.append(element)

    def solve(self):
        model = cp_model.CpModel()
        for el in self.elements:
            el.add_to_model(self.time_horizon, model)

        self.v_power = [
            sum([el.v_power[ts] for el in self.elements])
            for ts in range(self.time_horizon)
        ]

        for tp in self.v_power:
            model.add(tp <= self.max_power)
            model.add(tp >= -self.max_power)

        self.v_costs = [
            self.v_power[ts] * self.prices[ts] for ts in range(self.time_horizon)
        ]
        model.minimize(sum(self.v_costs))

        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = 10
        res = solver.solve(model, solution_callback=cp_model.ObjectiveSolutionPrinter())

        print(f"Status: {solver.StatusName(res)}")
        if res == cp_model.OPTIMAL or res == cp_model.FEASIBLE:
            print(f"Objective value: {solver.ObjectiveValue():.2f} EUR")

        for el in self.elements:
            el.evaluate(solver)

        self.power = np.array([solver.value(p) for p in self.v_power])
        self.costs = np.array([solver.float_value(c) for c in self.v_costs])

# Utils

In [ ]:
def ts2i(hour, minute=0):
    return hour * 4 + minute // 15


def load_prices(start, end):
    """Prices for given period, EUR/kWh"""
    df = pd.read_csv("data/nordpool-lv.csv", parse_dates=True, index_col=0)
    df = df[(df.index >= start) & ((df.index < end))]
    df = df.sort_index()
    return df["price"]


def load_avg_use(hours, daily_consumption):
    """Daily consumption profile in Wh/15min"""
    df = pd.read_excel(
        "data/DailyPattern.xlsx", sheet_name="DayPatternHour", index_col=0, nrows=24
    )
    df = df["Consumption%"]
    df = df.rename("avg")
    df = df.sort_index()
    assert df.sum() == 100

    df = df / 100 * daily_consumption / 4
    return df[hours].astype(int)

def load_solar(idx, size_m2):  
    df = pd.read_csv("data/solar_tmy.csv", skiprows=17, nrows=24 * 365)

    df.index = pd.to_datetime(df["time(UTC)"], format="%Y%m%d:%H%M").rename("ts")
    df["year-hour"] = df.index.strftime("%m-%d %H:%M")

    df = pd.merge(
        df,
        pd.Series(idx, name="ts"),
        left_on=df.index.strftime("%m-%d %H:%M"),
        right_on=idx.strftime("%m-%d %H:%M"),
    )

    df = df.set_index("ts")["G(h)"].rename("ghi")
    df = df.resample("15min").interpolate()

    df = pd.Series(index=idx.rename("ts"), data=df).fillna(0)
    return (df * size_m2).astype(int)

# Setup

In [ ]:
prices = load_prices("2026-01-12 12:00", "2026-01-14")
avg_profile = load_avg_use(prices.index.hour, daily_consumption=10_000)
solar_ghi = load_solar(prices.index, size_m2=16)

daily_average = FixedDevice("Daily average", list(avg_profile))
pv_panels = FixedDevice("PV panels", list(-solar_ghi))

In [ ]:
THIS_DAY = -12
NEXT_DAY = 12

devices = [
    Device(
        "Washing",
        [800, 2200, 1000, 1800, 800, 1000],
        lim_start=ts2i(THIS_DAY + 15),
        lim_end=ts2i(THIS_DAY + 23),
    ),
    Device(
        "Washing",
        [800, 2200, 1000, 1800, 800, 1000],
        lim_start=ts2i(NEXT_DAY + 15),
        lim_end=ts2i(NEXT_DAY + 23),
    ),
    Device(
        "Computer",
        [500] * ts2i(5),
        lim_start=ts2i(THIS_DAY + 12),
        lim_end=ts2i(THIS_DAY + 18),
    ),
    Device(
        "Computer",
        [500] * ts2i(8),
        lim_start=ts2i(NEXT_DAY + 8),
        lim_end=ts2i(NEXT_DAY + 18),
    ),
    Device(
        "Cooking",
        [2200] * ts2i(1, 30),
        lim_start=ts2i(THIS_DAY + 17),
        lim_end=ts2i(THIS_DAY + 21),
    ),
    Device(
        "Cooking",
        [2200] * ts2i(0, 45),
        lim_start=ts2i(NEXT_DAY + 6, 30),
        lim_end=ts2i(NEXT_DAY + 8, 30),
    ),
    Device(
        "Cooking",
        [2200] * ts2i(1, 30),
        lim_start=ts2i(NEXT_DAY + 17),
        lim_end=ts2i(NEXT_DAY + 21),
    ),
]

In [ ]:
battery = Battery(
        "battery",
        capacity=50000,
        max_inflow=3500,
        max_outflow=-5000,
        start_level=20000,
        end_level=30000,
    )

# Plots

In [ ]:
from itertools import groupby
from matplotlib.dates import DateFormatter, DateLocator, HourLocator
from matplotlib.patches import Patch


def plot_prices(ax, ts, y):
    ax.step(ts, y, where="post")
    ax.set_ylabel("Price, $EUR/kWh$")


def plot_power(ax, ts, y, w=1 / 24 / 4):
    ax.bar(
        ts,
        y,
        width=w,
        color=["C3" if p > 0 else "C2" for p in y],
        zorder=2,
        alpha=0.75,
        align="edge",
    )
    ax.set_ylabel("Power profile, $kWh$")


def plot_costs(ax, ts, y):
    ax.step(ts, y, where="post")
    ax.annotate(
        f"{y[-1]:.2f} EUR",
        xy=(ts[-1], y[-1]),
        xytext=(0.9, 0.8),
        textcoords="axes fraction",
        arrowprops=dict(arrowstyle="->"),
    )
    ax.set_ylabel("Cumulative costs, $EUR$")


def plot_battery(ax, ts, y, w=1 / 24 / 4):
    ax.bar(
        ts,
        y,
        width=w,
        color="C1",
        zorder=2,
        alpha=0.75,
        align="edge",
    )
    ax.set_ylabel("Battery charge, $kWh$")


def plot_devices(ax, ts, devices: list[Device]):
    num_groups = len(set(dev.name for dev in devices))
    handles = []
    for i, (name, grp) in enumerate(groupby(devices, lambda d: d.name)):
        y = num_groups - i
        clr = f"C{i}"
        handles.append(Patch(color=clr, label=name))

        for dev in grp:
            a = ts[dev.lim_start if dev.lim_start else 0]
            if dev.lim_end and dev.lim_end < len(ts):
                b = ts[dev.lim_end]
            else:
                b = ts[-1] + pd.Timedelta(minutes=15)
            ax.plot([a, b], [y, y], "-|", color=clr)

            ax.barh(
                y,
                width=dev.len / 24 / 4,
                left=ts[dev.start],
                alpha=0.75,
                zorder=2,
                color=clr,
            )
    ax.set_ylabel("Device schedules")
    ax.set_yticklabels([])
    ax.set_ylim(0, num_groups + 1)
    ax.legend(handles=handles, ncols=4, loc="lower center", bbox_to_anchor=(0.5, -0.5))


def plot_common(ax, ts_min, ts_max):
    ax.grid()
    ax.set_xlim(
        ts_min - pd.Timedelta(minutes=30),
        ts_max + pd.Timedelta(minutes=30),
    )
    ax.xaxis.set_tick_params(which="both", labelbottom=True)
    ax.xaxis.set_major_formatter(DateFormatter("%H:%M"))
    ax.xaxis.set_major_locator(HourLocator(range(0, 24, 3)))
    ax.xaxis.set_minor_locator(HourLocator())

# Models

## Only appliences

In [ ]:
hm = HouseholdModel(max_power=5200, prices=list(prices / 1000 / 4))
for dev in devices:
    hm.add(dev)
hm.solve()

In [ ]:
fig, axs = plt.subplots(nrows=4, sharex=True, figsize=(10, 8))

ts = prices.index
plot_prices(axs[0], ts, prices)
plot_power(axs[1], ts, np.array(hm.power) / 1000)
plot_costs(axs[2], ts, np.cumsum(hm.costs))
plot_devices(axs[3], ts, devices)

for ax in axs:
    plot_common(ax, ts[0], ts[-1])

fig.savefig("img/1. Home appliances.png", dpi=120, bbox_inches="tight")

## Battery only

In [ ]:
hm = HouseholdModel(max_power=5200, prices=list(prices / 1000 / 4))
hm.add(battery)
hm.solve()

In [ ]:
fig, axs = plt.subplots(nrows=4, sharex=True, figsize=(10, 8))

ts = prices.index
plot_prices(axs[0], ts, prices)
plot_power(axs[1], ts, np.array(hm.power) / 1000)
plot_costs(axs[2], ts, np.cumsum(hm.costs))
plot_battery(axs[3], ts, np.array(battery.level) / 1000)

for ax in axs:
    plot_common(ax, ts[0], ts[-1])

fig.savefig("img/2. Battery.png", dpi=120, bbox_inches="tight")

## Battery and daily average

In [ ]:
hm = HouseholdModel(max_power=5200, prices=list(prices / 1000 / 4))
hm.add(battery)
hm.add(daily_average)
hm.solve()

In [ ]:
fig, axs = plt.subplots(nrows=5, sharex=True, figsize=(10, 10))

ts = prices.index
plot_prices(axs[0], ts, prices)
plot_power(axs[1], ts, np.array(hm.power) / 1000)
plot_costs(axs[2], ts, np.cumsum(hm.costs))
plot_battery(axs[3], ts, np.array(battery.level) / 1000)
plot_power(axs[4], ts, np.array(daily_average.power) / 1000)
axs[4].set_ylabel("Daily profile, $kWh$")

for ax in axs:
    plot_common(ax, ts[0], ts[-1])

fig.savefig("img/3. Daily average.png", dpi=120, bbox_inches="tight")

## All together

In [ ]:
hm = HouseholdModel(max_power=5200, prices=list(prices / 1000 / 4))
for dev in devices:
    hm.add(dev)
hm.add(daily_average)
hm.add(battery)
hm.solve()

In [ ]:
fig, axs = plt.subplots(nrows=5, sharex=True, figsize=(10, 10))

ts = prices.index
plot_prices(axs[0], ts, prices)
plot_power(axs[1], ts, np.array(hm.power) / 1000)
plot_costs(axs[2], ts, np.cumsum(hm.costs))
plot_battery(axs[3], ts, np.array(battery.level) / 1000)
plot_devices(axs[4], ts, devices)

for ax in axs:
    plot_common(ax, ts[0], ts[-1])

fig.savefig("img/4. All together.png", dpi=120, bbox_inches="tight")

## Daily average and PV panels

In [ ]:
hm = HouseholdModel(max_power=5200, prices=list(prices / 1000 / 4))
hm.add(daily_average)
hm.add(battery)
hm.add(pv_panels)
hm.solve()

In [ ]:
fig, axs = plt.subplots(nrows=5, sharex=True, figsize=(10, 10))

ts = prices.index
plot_prices(axs[0], ts, prices)
plot_power(axs[1], ts, np.array(hm.power) / 1000)
plot_costs(axs[2], ts, np.cumsum(hm.costs))
plot_battery(axs[3], ts, np.array(battery.level) / 1000)
plot_power(axs[4], ts, np.array(daily_average.power) / 1000)
plot_power(axs[4], ts, np.array(pv_panels.power) / 1000)

axs[4].set_ylabel("A+/A-, $kWh$")

for ax in axs:
    plot_common(ax, ts[0], ts[-1])

fig.savefig("img/5. Daily average and PV.png", dpi=120, bbox_inches="tight")

In [ ]:
df = pd.DataFrame(prices)
df["power"] = np.array(hm.power) / 1000
df["costs"] = np.cumsum(hm.costs)
df["bat_power"] = np.array(battery.power) / 1000
df["bat_level"] = np.array(battery.level) / 1000
df["daily_avg"] = np.array(daily_average.power) / 1000
df["pv_panels"] = np.array(pv_panels.power) / 1000

df.to_csv("img/5. Daily average and PV.csv")
df

# Multiple days

In [ ]:
res = pd.DataFrame(
    index=pd.date_range(
        "2026-01-12 12:00", "2026-01-20", freq="15min", inclusive="left"
    )
)

curr_level = 20_000
curr_costs = 0.0

for i in range(7):
    print(f"=== DAY {i+1} ===")
    day_from = pd.Timestamp("2026-01-12 12:00") + pd.Timedelta(days=i)
    day_to = pd.Timestamp("2026-01-14") + pd.Timedelta(days=i)
    prices = load_prices(day_from, day_to)

    battery.start_level = curr_level
    hm = HouseholdModel(max_power=5200, prices=list(prices / 1000 / 4))
    for dev in devices:
        hm.add(dev)
    hm.add(daily_average)
    hm.add(battery)
    hm.solve()

    print("=== ", curr_costs)

    res.loc[prices.index, "price"] = prices
    res[f"costs {i+1}"] = pd.Series(data=hm.costs, index=prices.index).cumsum() + curr_costs 
    res[f"level {i+1}"] = pd.Series(data=battery.level, index=prices.index)

    curr_level = battery.level[24*4]
    curr_costs += sum(hm.costs[:24*4])

In [ ]:
from matplotlib.dates import DayLocator


fig, axs = plt.subplots(nrows=3, sharex=True, figsize=(10, 6))

plot_prices(axs[0], res.index, res["price"])

axs[0].set_xlim(
    res.index[0] - pd.Timedelta(hours=1),
    res.index[-1] + pd.Timedelta(hours=1),
)
axs[1].set_ylabel("Cumulative costs, $EUR$")
axs[2].set_ylabel("Battery charge, $kWh$")
for i in range(1, 8):
    axs[1].step(res.index, res[f"costs {i}"], where="post")
    axs[2].step(res.index, res[f"level {i}"] / 1000, where="post")


for ax in axs:
    ax.xaxis.set_tick_params(which="both", labelbottom=True)
    ax.xaxis.set_major_formatter(DateFormatter("%d-%b"))
    ax.xaxis.set_major_locator(DayLocator(interval=1))
    ax.xaxis.set_minor_locator(HourLocator([0, 6, 12, 18]))
    ax.grid(which="major")
    ax.grid(which="minor", alpha=0.33)

fig.savefig("img/6. Multiple days.png", dpi=120, bbox_inches="tight")